## Build the Experiment

- uses the 01-Translation folder experiment files to build the html files for each experiment
- uses the 02-Stimuli folder to grab the stimuli lists for each task 
- creates the separate .jas files for each experiment
- creates the .jzip for importing 



## Libraries and Functions

In [54]:
import os
import re
import json
import shutil
from pathlib import Path
import pandas as pd
import json
import re
import shutil

DEFAULT_INDEX_TEMPLATES = (
    "index_consent_demos.html",
    "index_page1.html",
    "index_page2.html",
    "index_page3.html",
    "index_page4.html",
    "index_page5.html",
    "index_page6.html",
    "index_page7.html",
    "index_page8.html",
    "index_page9.html",
    "index_page10.html",
    "thank_you.html",
)

def translate_logic_strings(html, translation_dict):

    pattern = r'(===\s*)(["\'])(.*?)(\2)'

    def replace_match(match):
        prefix = match.group(1)
        quote = match.group(2)
        inner = match.group(3)
        end_quote = match.group(4)

        # Only translate if it's in dictionary
        if inner in translation_dict:
            tgt = translation_dict[inner]

            # 🔥 REMOVE inner quotes here
            tgt = tgt.replace('"', '')
            return f'{prefix}{quote}{tgt}{end_quote}'

        return match.group(0)

    html = re.sub(pattern, replace_match, html)

    return html

def fix_next_only(html):

    # -------------------------
    # 1) Fix JATOS function
    # jatos.start<something>Component → jatos.startNextComponent
    # -------------------------
    html = re.sub(
        r'jatos\.start[^C]*Component',
        'jatos.startNextComponent',
        html
    )

    # -------------------------
    # 2) Fix corrupted key names
    # page<something>Text → pageNextText (ONLY if it was supposed to be Next)
    # -------------------------
    html = re.sub(
        r'page[^P]*Text',
        'pageNextText',
        html
    )

    return html

def inject_bkey(html, bkey):
    return re.sub(
        r'(const\s+bkey\s*=\s*)\d+;',
        rf'\g<1>{bkey};',
        html
    )

def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def load_translation_dict(csv_path, english_col="English", translation_col="translation"):
    df = pd.read_csv(csv_path)

    if english_col not in df.columns:
        english_col = df.columns[0]

    if translation_col not in df.columns:
        # fallback: second column
        non_eng = [c for c in df.columns if c != english_col]
        if not non_eng:
            raise ValueError("Could not find translation column in translation CSV")
        translation_col = non_eng[0]

    df = df[[english_col, translation_col]].dropna()
    df[english_col] = df[english_col].astype(str).str.strip()
    df[translation_col] = df[translation_col].astype(str).str.strip()

    return dict(zip(df[english_col], df[translation_col]))

def _normalize_quotes(text):
    """Normalize curly/smart quotes to straight quotes for matching."""
    return (
        text.replace('‘', "'").replace('’', "'")
            .replace('“', '"').replace('”', '"')
    )

def replace_in_text(text, translation_dict):
    # Normalize curly quotes so template curly apostrophes match CSV straight ones
    text = _normalize_quotes(text)
    items = sorted(translation_dict.items(), key=lambda x: -len(x[0]))
    for src, tgt in items:
        if not src or not tgt:
            continue
        tgt = tgt.replace('"', '')
        pattern = re.compile(rf"(?<!\w){re.escape(src)}(?!\w)", flags=re.DOTALL)
        text = pattern.sub(tgt, text)
    return text


def apply_translations(html, translation_dict):

    # -------------------------
    # 1) Translate html: `...` (SurveyJS HTML blocks)
    # -------------------------
    template_pattern = r'(html\s*:\s*)`([\s\S]*?)`'

    def replace_template(match):
        prefix = match.group(1)
        inner = match.group(2)
        new_inner = replace_in_text(inner, translation_dict)
        return f"{prefix}`{new_inner}`"

    html = re.sub(template_pattern, replace_template, html)

    # -------------------------
    # 2) Translate choices arrays
    # -------------------------
    choices_pattern = r'(\bchoices\s*:\s*\[)([\s\S]*?)(\])'

    def replace_choices(match):
        start = match.group(1)
        body = match.group(2)
        end = match.group(3)

        string_pattern = r'(["\'])(.*?)(\1)'

        def replace_choice_string(m):
            quote = m.group(1)
            inner = m.group(2)
            end_quote = m.group(3)
            new_inner = replace_in_text(inner, translation_dict)
            return f"{quote}{new_inner}{end_quote}"

        new_body = re.sub(string_pattern, replace_choice_string, body)
        return f"{start}{new_body}{end}"

    html = re.sub(choices_pattern, replace_choices, html)

    # -------------------------
    # 3) Translate title fields ONLY
    # -------------------------
    title_pattern = r'(\btitle\s*:\s*)(["\'])(.*?)(\2)'

    def replace_title(match):
        prefix = match.group(1)
        quote = match.group(2)
        inner = match.group(3)
        end_quote = match.group(4)
        new_inner = replace_in_text(inner, translation_dict)
        return f"{prefix}{quote}{new_inner}{end_quote}"

    html = re.sub(title_pattern, replace_title, html)

    # -------------------------
    # 4) Translate simple property values (JS object syntax: key: "value")
    # -------------------------
    prop_pattern = r'(\b[A-Za-z_][A-Za-z0-9_]*\b\s*:\s*)(["\'])(.*?)(\2)'

    def replace_property(match):
        prefix = match.group(1)
        quote = match.group(2)
        inner = match.group(3)
        end_quote = match.group(4)

        if inner in translation_dict:
            new_inner = translation_dict[inner]
            return f"{prefix}{quote}{new_inner}{end_quote}"

        return match.group(0)

    html = re.sub(prop_pattern, replace_property, html)

    # -------------------------
    # 5) Translate HTML attribute values (attr="value")
    # -------------------------
    html_attr_pattern = r'(\b[A-Za-z_][A-Za-z0-9_-]*\b\s*=\s*)(["\'])(.*?)(\2)'

    def replace_html_attr(match):
        prefix = match.group(1)
        quote = match.group(2)
        inner = match.group(3)
        end_quote = match.group(4)

        if inner in translation_dict:
            new_inner = translation_dict[inner]
            new_inner = new_inner.replace('"', '')
            return f"{prefix}{quote}{new_inner}{end_quote}"

        return match.group(0)

    html = re.sub(html_attr_pattern, replace_html_attr, html)

    # -------------------------
    # 6) Translate HTML text between tags
    # -------------------------
    html_text_pattern = r'>([^<>]+)<'

    def replace_html_text(match):
        inner = match.group(1)

        if not inner.strip():
            return match.group(0)

        new_inner = replace_in_text(inner, translation_dict)
        return f">{new_inner}<"

    # Apply repeatedly until no more changes (handles nesting)
    prev_html = None
    while prev_html != html:
        prev_html = html
        html = re.sub(html_text_pattern, replace_html_text, html)

    # -------------------------
    # 7) Translate inline UI messages (e.g., error messages)
    # -------------------------
    ui_string_pattern = r'(["\'])([^"\']{5,})(\1)'

    def replace_ui_string(match):
        quote = match.group(1)
        inner = match.group(2)
        end_quote = match.group(3)

        # heuristic: translate only sentences (contains space)
        if " " not in inner:
            return match.group(0)

        new_inner = replace_in_text(inner, translation_dict)

        # if nothing changed, keep original
        if new_inner == inner:
            return match.group(0)

        return f"{quote}{new_inner}{end_quote}"

    html = re.sub(ui_string_pattern, replace_ui_string, html)

    return html


def insert_words_into_html_string(html, word_list, var_name="FULL_WORD_LIST"):
    words_js = json.dumps(word_list, ensure_ascii=False, indent=2)
    replacement = f"const {var_name} = {words_js};"

    pattern = rf"const\s+{re.escape(var_name)}\s*=\s*\[.*?\];"
    new_html, n = re.subn(pattern, replacement, html, flags=re.DOTALL)

    if n == 0:
        return html

    return new_html


def load_word_list(csv_path):
    df = pd.read_csv(csv_path)
    if "word" not in df.columns:
        raise ValueError(f"{csv_path} must contain a 'word' column")

    return (
        df["word"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .tolist()
    )


def copy_static_files(src_dir, dst_dir, exclude=None):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    if exclude is None:
        exclude = {"index.html", "consent.html"}
    else:
        exclude = set(exclude)

    for item in src_dir.iterdir():
        if item.name in exclude:
            continue

        target = dst_dir / item.name
        if item.is_dir():
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)


def build_task_htmls_from_translation_csv(
    lang,
    translation_csv,
    task_template_root="03-Tasks",
    stimuli_root="02-Stimuli",
    output_root="03-Tasks",
    tasks=("aoa", "image", "concrete", "valence", "arousal", "familiar"),
    index_templates=DEFAULT_INDEX_TEMPLATES,
    template_output_map=None,
    bkey=None,
):
    translation_dict = load_translation_dict(translation_csv)

    summary = []

    if isinstance(index_templates, str):
        index_templates = (index_templates,)
    index_templates = tuple(index_templates)

    if template_output_map is None:
        template_output_map = {}

    original_output_map = dict(template_output_map)
    template_output_map = {
        tpl: original_output_map.get(tpl, tpl) for tpl in index_templates
    }

    for task in tasks:
        task_dir = Path(task_template_root) / task
        consent_path = task_dir / "consent.html"

        if not consent_path.exists():
            raise FileNotFoundError(f"Missing consent template: {consent_path}")

        base_consent_html = read_text(consent_path)

        translated_consent_html = apply_translations(base_consent_html, translation_dict)
        translated_consent_html = fix_next_only(translated_consent_html)

        translated_templates = {}
        for template_name in index_templates:
            template_path = task_dir / template_name
            if not template_path.exists():
                continue

            base_template_html = read_text(template_path)
            translated_template = apply_translations(
                base_template_html, translation_dict
            )
            translated_template = fix_next_only(translated_template)
            translated_template = translate_logic_strings(
                translated_template, translation_dict
            )
            if bkey is not None:
                translated_template = inject_bkey(translated_template, bkey)
            translated_templates[template_name] = translated_template

        if not translated_templates:
            raise FileNotFoundError(
                f"Missing task templates for {task}: {index_templates}"
            )

        stimuli_dir = Path(stimuli_root) / lang / task
        if not stimuli_dir.exists():
            raise FileNotFoundError(f"Missing stimuli folder: {stimuli_dir}")

        stimuli_files = sorted(stimuli_dir.glob("*.csv"))
        if not stimuli_files:
            print(f"No stimuli files found for {task} in {stimuli_dir}")
            continue

        for stimuli_file in stimuli_files:
            words = load_word_list(stimuli_file)

            exp_name = stimuli_file.stem
            exp_dir = Path(output_root) / lang / task / exp_name
            exp_dir.mkdir(parents=True, exist_ok=True)

            # copy assets first
            copy_static_files(
                task_dir, exp_dir, exclude=list(translated_templates.keys()) + ["consent.html"]
            )

            # insert task words
            for template_name, template_html in translated_templates.items():
                filled_template_html = insert_words_into_html_string(
                    template_html, words
                )
                output_name = template_output_map.get(template_name, template_name)
                write_text(exp_dir / output_name, filled_template_html)

            write_text(exp_dir / "consent.html", translated_consent_html)

            summary.append(
                {
                    "task": task,
                    "stimuli_file": str(stimuli_file),
                    "output_dir": str(exp_dir),
                    "template": ", ".join(sorted(translated_templates.keys())),
                    "n_words": len(words),
                }
            )

    return pd.DataFrame(summary)

### Build Ukr

In [55]:
summary_df = build_task_htmls_from_translation_csv(
    lang="uk",
    translation_csv="../01-Translation/05_final_languages/uk/uk_experiment.csv",
    task_template_root="../03-Tasks",
    stimuli_root="../02-Stimuli",
    output_root="../03-Tasks/builds",
    # bkey = 98063, # erin
    bkey = 97987 # addie 
)

summary_df

,task,stimuli_file,output_dir,template,n_words
0,aoa,../02-Stimuli/uk/aoa/aoa_list_1.csv,../03-Tasks/builds/uk/aoa/aoa_list_1,"index_consent_demos.html, index_page1.html, in...",200
1,aoa,../02-Stimuli/uk/aoa/aoa_list_10.csv,../03-Tasks/builds/uk/aoa/aoa_list_10,"index_consent_demos.html, index_page1.html, in...",139
2,aoa,../02-Stimuli/uk/aoa/aoa_list_2.csv,../03-Tasks/builds/uk/aoa/aoa_list_2,"index_consent_demos.html, index_page1.html, in...",200
3,aoa,../02-Stimuli/uk/aoa/aoa_list_3.csv,../03-Tasks/builds/uk/aoa/aoa_list_3,"index_consent_demos.html, index_page1.html, in...",200
4,aoa,../02-Stimuli/uk/aoa/aoa_list_4.csv,../03-Tasks/builds/uk/aoa/aoa_list_4,"index_consent_demos.html, index_page1.html, in...",200
5,aoa,../02-Stimuli/uk/aoa/aoa_list_5.csv,../03-Tasks/builds/uk/aoa/aoa_list_5,"index_consent_demos.html, index_page1.html, in...",200
6,aoa,../02-Stimuli/uk/aoa/aoa_list_6.csv,../03-Tasks/builds/uk/aoa/aoa_list_6,"index_consent_demos.html, index_page1.html, in...",200
7,aoa,../02-Stimuli/uk/aoa/aoa_list_7.csv,../03-Tasks/builds/uk/aoa/aoa_list_7,"index_consent_demos.html, index_page1.html, in...",200
8,aoa,../02-Stimuli/uk/aoa/aoa_list_8.csv,../03-Tasks/builds/uk/aoa/aoa_list_8,"index_consent_demos.html, index_page1.html, in...",200
9,aoa,../02-Stimuli/uk/aoa/aoa_list_9.csv,../03-Tasks/builds/uk/aoa/aoa_list_9,"index_consent_demos.html, index_page1.html, in...",200


## Update JAS File

In [56]:
import json
import uuid
from pathlib import Path


def update_jas_file(jas_path, new_title, new_dir_name=None):
    jas_path = Path(jas_path)

    with open(jas_path, "r", encoding="utf-8") as f:
        jas = json.load(f)

    if "data" not in jas:
        raise ValueError(f"{jas_path} does not have expected JATOS .jas structure")

    data = jas["data"]

    # study-level fields
    data["uuid"] = str(uuid.uuid4())
    data["title"] = new_title

    if new_dir_name is None:
        new_dir_name = new_title
    data["dirName"] = new_dir_name

    # component-level fields
    for i, comp in enumerate(data.get("componentList", []), start=1):
        comp["uuid"] = str(uuid.uuid4())
        comp["title"] = new_title
        # usually keep htmlFilePath = index.html

    # batch-level fields
    for batch in data.get("batchList", []):
        batch["uuid"] = str(uuid.uuid4())
        # you can leave title as "Default" if you want

    with open(jas_path, "w", encoding="utf-8") as f:
        json.dump(jas, f, indent=2, ensure_ascii=False)


def update_all_jas_files(base_dir):
    base_dir = Path(base_dir)
    jas_files = sorted(base_dir.rglob("*.jas"))

    print(f"Found {len(jas_files)} .jas files")

    for jas_path in jas_files:
        # expected structure:
        # .../<lang>/<task>/<experiment_folder>/<something>.jas
        try:
            lang = jas_path.parts[-4]
            task = jas_path.parts[-3]
            exp = jas_path.parts[-2]
            new_title = f"{lang}_{task}_{exp}"
            new_dir_name = exp
        except Exception:
            new_title = jas_path.stem
            new_dir_name = jas_path.stem

        update_jas_file(jas_path, new_title=new_title, new_dir_name=new_dir_name)
        print(f"Updated {jas_path} -> title={new_title}, dirName={new_dir_name}")

In [57]:

update_all_jas_files("../03-Tasks/builds/uk")

Found 60 .jas files
Updated ../03-Tasks/builds/uk/aoa/aoa_list_1/aoa_components12534479281580246847.jas -> title=uk_aoa_aoa_list_1, dirName=aoa_list_1
Updated ../03-Tasks/builds/uk/aoa/aoa_list_10/aoa_components12534479281580246847.jas -> title=uk_aoa_aoa_list_10, dirName=aoa_list_10
Updated ../03-Tasks/builds/uk/aoa/aoa_list_2/aoa_components12534479281580246847.jas -> title=uk_aoa_aoa_list_2, dirName=aoa_list_2
Updated ../03-Tasks/builds/uk/aoa/aoa_list_3/aoa_components12534479281580246847.jas -> title=uk_aoa_aoa_list_3, dirName=aoa_list_3
Updated ../03-Tasks/builds/uk/aoa/aoa_list_4/aoa_components12534479281580246847.jas -> title=uk_aoa_aoa_list_4, dirName=aoa_list_4
Updated ../03-Tasks/builds/uk/aoa/aoa_list_5/aoa_components12534479281580246847.jas -> title=uk_aoa_aoa_list_5, dirName=aoa_list_5
Updated ../03-Tasks/builds/uk/aoa/aoa_list_6/aoa_components12534479281580246847.jas -> title=uk_aoa_aoa_list_6, dirName=aoa_list_6
Updated ../03-Tasks/builds/uk/aoa/aoa_list_7/aoa_components1

## Zip up the Files

In [58]:
import zipfile
from pathlib import Path

def zip_all_experiments(
    base_dir, index_templates=DEFAULT_INDEX_TEMPLATES, template_output_map=None
):
    base_dir = Path(base_dir)

    if isinstance(index_templates, str):
        index_templates = (index_templates,)
    index_templates = tuple(index_templates)

    if isinstance(template_output_map, str):
        template_output_map = {template_output_map: template_output_map}
    elif template_output_map is None:
        template_output_map = {}

    expected_files = set(index_templates) | set(template_output_map.values())

    exp_dirs = [
        d
        for d in base_dir.rglob("*")
        if any((d / template_name).exists() for template_name in expected_files)
    ]

    print(f"Found {len(exp_dirs)} experiment folders")

    for exp_dir in exp_dirs:
        jas_files = list(exp_dir.glob("*.jas"))
        if not jas_files:
            print(f"Skipping {exp_dir}: no .jas file found")
            continue

        jas_path = jas_files[0]
        jzip_path = exp_dir.parent / f"{exp_dir.name}.jzip"

        with zipfile.ZipFile(jzip_path, "w", zipfile.ZIP_DEFLATED) as z:
            # 1) add the study directory contents under the folder name
            for file in exp_dir.rglob("*"):
                if not file.is_file():
                    continue
                if file.name.startswith("."):
                    continue
                if "__MACOSX" in file.parts:
                    continue

                arcname = Path(exp_dir.name) / file.relative_to(exp_dir)
                z.write(file, arcname)

            # 2) add the .jas file at zip root
            z.write(jas_path, jas_path.name)

        print(f"Created: {jzip_path}")








In [59]:

zip_all_experiments(
    "../03-Tasks/builds/uk",
)

Found 60 experiment folders
Created: ../03-Tasks/builds/uk/concrete/concrete_list_10.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_1.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_6.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_8.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_9.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_7.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_5.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_2.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_3.jzip
Created: ../03-Tasks/builds/uk/concrete/concrete_list_4.jzip
Created: ../03-Tasks/builds/uk/image/image_list_10.jzip
Created: ../03-Tasks/builds/uk/image/image_list_2.jzip
Created: ../03-Tasks/builds/uk/image/image_list_5.jzip
Created: ../03-Tasks/builds/uk/image/image_list_4.jzip
Created: ../03-Tasks/builds/uk/image/image_list_3.jzip
Created: ../03-Tasks/builds/uk/image/image_list_8.jzip
Created: ../03-Tasks/builds/uk

## Build Priming Experiment JSON

Translates the UI text in `spaml_template.json` (instructions, breaks, consent, end screen)
and saves a language-patched copy as `builds/{lang}/priming/{lang}_spaml.json`.

This translated JSON is shared across all panel versions — versions differ only in their
embedded stimulus hash files, which are built separately by `build_priming_stimuli.R`.

In [ ]:
CHECKED_TAGS = ["b", "kbd", "strong", "em", "i"]

def fix_tag_corruption(html):
    """Fix common HTML tag corruptions introduced by machine translation."""

    # <unk> tokens → closing </kbd> (most common MT artifact)
    html = re.sub(r'<unk>', '</kbd>', html)

    # truncated <kb> → <kbd> (first), then paired <kbd>...<kbd> → <kbd>...</kbd> below
    html = re.sub(r'<kb>', '<kbd>', html)
    html = re.sub(r'</kb>', '</kbd>', html)

    # '} kbd>' (template closing brace + space + missing '</')  → '}</kbd>'
    html = re.sub(r'} kbd>', '}</kbd>', html)

    # </kbd>text</kbd> → <kbd>text</kbd>  (MT added '/' to opening tag)
    html = re.sub(r'</kbd>([^<]{1,30})</kbd>', r'<kbd>\1</kbd>', html)

    for tag in CHECKED_TAGS:
        # <tag>...<tag> → <tag>...</tag>
        # Permissive match: content may contain other tags, just not the same tag
        html = re.sub(
            rf'(<{tag}>)((?:(?!</?{tag}>).)*?)(<{tag}>)',
            rf'\1\2</{tag}>',
            html,
            flags=re.DOTALL,
        )

    # Remove stray opening inline tags inside <kbd>...</kbd> blocks
    # e.g. <kbd>$<b></kbd> → <kbd>$</kbd>
    for inner_tag in CHECKED_TAGS:
        html = re.sub(
            rf'(<kbd>[^<]*?)<{inner_tag}>([^<]*?</kbd>)',
            r'\1\2',
            html,
        )

    return html


def _norm_tmpl(expr):
    """Normalize template expression spacing for comparison: '${ window.x }' == '${window.x}'."""
    return re.sub(r'\s+', '', expr)


def restore_kbd_exprs(translated, original):
    """Re-inject template expressions that the CSV translation dropped from <kbd> tags."""
    # Collect template expressions that were inside <kbd> in the original (in order)
    expected = [
        m2.group(0)
        for m in re.finditer(r'<kbd>(.*?)</kbd>', original, re.DOTALL)
        for m2 in re.finditer(r'\$\{[^}]+\}', m.group(1))
    ]
    if not expected:
        return translated

    # Normalize-compare: treat '${ window.word_key }' and '${window.word_key}' as the same
    present_norm = {_norm_tmpl(e) for e in re.findall(r'\$\{[^}]+\}', translated)}
    missing = [e for e in expected if _norm_tmpl(e) not in present_norm]

    if not missing:
        return translated

    expr_q = list(missing)

    def inject(m):
        inner = m.group(1).strip()
        if re.search(r'\$\{', inner):   # already has a template expr → skip
            return m.group(0)
        if inner in ('Space', 'Space Bar'):  # keyboard shortcut label → skip
            return m.group(0)
        if not expr_q:
            return m.group(0)
        return f'<kbd>{expr_q.pop(0)}</kbd>'

    return re.sub(r'<kbd>(.*?)</kbd>', inject, translated, flags=re.DOTALL)


def warn_unbalanced_tags(html, label=""):
    """Print a warning if any checked tags are unbalanced."""
    problems = []
    for tag in CHECKED_TAGS:
        opens  = len(re.findall(rf'<{tag}(?:\s[^>]*)?>',  html))
        closes = len(re.findall(rf'</{tag}>',              html))
        if opens != closes:
            problems.append(f"<{tag}> opens={opens} closes={closes}")
    if problems:
        print(f"  ⚠ tag mismatch{' in ' + label if label else ''}: {', '.join(problems)}")


def build_priming_spaml(
    lang,
    translation_csv=None,
    template_path="semantic_priming/spaml_template.json",
    output_root="builds",
):
    if translation_csv is None:
        translation_csv = f"../01-Translation/05_final_languages/{lang}/{lang}_experiment.csv"

    translation_dict = load_translation_dict(translation_csv)

    with open(template_path, "r", encoding="utf-8") as f:
        template = json.load(f)

    patched = json.loads(json.dumps(template))  # deep copy

    translated = 0
    for key, comp in patched.get("components", {}).items():
        comp_type = comp.get("type", "")
        label = f"component {key} ({comp.get('title', '')})"

        # Translate HTML content (Screen and Form types)
        if comp_type in ("lab.html.Screen", "lab.html.Form"):
            original = comp.get("content", "")
            if original:
                content = replace_in_text(original, translation_dict)
                content = apply_translations(content, translation_dict)
                content = fix_tag_corruption(content)
                content = restore_kbd_exprs(content, original)
                warn_unbalanced_tags(content, label)
                comp["content"] = content
                translated += 1

        # Translate canvas frame footer bar (uses 'context' not 'content')
        # Normalize whitespace first so multi-line phrases match single-line CSV keys
        elif comp_type == "lab.canvas.Frame":
            original = comp.get("context", "")
            if original:
                normalized = re.sub(r'\s+', ' ', original).strip()
                context = replace_in_text(normalized, translation_dict)
                context = apply_translations(context, translation_dict)
                context = fix_tag_corruption(context)
                context = restore_kbd_exprs(context, normalized)
                warn_unbalanced_tags(context, label)
                comp["context"] = context
                translated += 1

        # Translate quoted string literals in JS messageHandler code
        for handler in comp.get("messageHandlers", []):
            code = handler.get("code", "")
            if not code:
                continue
            def _replace_js_str(m, td=translation_dict):
                q, inner, eq = m.group(1), m.group(2), m.group(3)
                return f"{q}{td.get(inner, inner)}{eq}"
            handler["code"] = re.sub(r"(['\"])([^'\"]+)(\1)", _replace_js_str, code)

    out_dir = Path(output_root) / lang / "priming"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{lang}_spaml.json"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(patched, f, ensure_ascii=False, indent=2)

    print(f"[{lang}] Translated {translated} components → {out_path}")

In [61]:
build_priming_spaml(lang="uk")

[uk] Translated 21 components → builds/uk/priming/uk_spaml.json
